In [21]:
import yaml
import pandas as pd
with open(r"C:\Users\hajjar\Desktop\code_caveat\caveat\configs\household_VAE.yaml") as f:
    config = yaml.safe_load(f)

df_s, df_a = pd.read_csv('observed_schedules.csv'), pd.read_csv('observed_attributes.csv')


pid_s = set(df_s['pid'].unique())
pid_a = set(df_a['pid'].unique())

pid_only_in_s = pid_a - pid_s
len(pid_only_in_s)

4997

 Chargement CSV...


FileNotFoundError: [Errno 2] No such file or directory: 'caveat/data/observed_schedules.csv'

In [24]:
for i,j in enumerate(["Work", "Leisure"]):
    print(i,j)

0 Work
1 Leisure


In [44]:
act_to_idx = {act:idx for idx,act in enumerate(["Work", "Leisure"],start=1)} | {"PAD":0}
idx_to_act = {idx:act for act,idx in act_to_idx.items()}

idx_to_act

{1: 'Work', 2: 'Leisure', 0: 'PAD'}

In [54]:
import pandas as pd

# ----------------------------
# Toy schedules (épisodes)
# ----------------------------
schedules = pd.DataFrame(
    [
        # Ménage 100 : pid 1
        {"pid": 1, "act": "Home",    "start": 0,   "end": 480},
        {"pid": 1, "act": "Work",    "start": 480, "end": 1020},
        {"pid": 1, "act": "Home",    "start": 1020,"end": 1440},

        # Ménage 100 : pid 2
        {"pid": 2, "act": "Home",    "start": 0,   "end": 540},
        {"pid": 2, "act": "School",  "start": 540, "end": 960},
        {"pid": 2, "act": "Leisure", "start": 960, "end": 1080},
        {"pid": 2, "act": "Home",    "start": 1080,"end": 1440},

        # Ménage 200 : pid 3
        {"pid": 3, "act": "Home",    "start": 0,   "end": 600},
        {"pid": 3, "act": "Shop",    "start": 600, "end": 720},
        {"pid": 3, "act": "Home",    "start": 720, "end": 1440},
    ]
)

# ----------------------------
# Toy attributes (lien pid -> ménage + attributs)
# ----------------------------
attributes = pd.DataFrame(
    [
        {"pid": 1, "hid": 100, "age": 41, "sex": "M"},
        {"pid": 2, "hid": 100, "age": 14, "sex": "F"},
        {"pid": 3, "hid": 200, "age": 33, "sex": "F"},
    ]
)


data = schedules[['pid', 'act', 'start', 'end']].merge(
            attributes[['pid', "hid"]], on='pid', how='left'
        )
for _, group in data.groupby("hid"):
    print("le",group)


100
200


In [59]:
unique_pids = [100,200,300,400,500]

if len(unique_pids) > 4:
    unique_pids = unique_pids[:4]
    print(unique_pids)

[100, 200, 300, 400]


In [3]:
import pandas as pd
import polars as pl
from pam import read
from pam.core import Population
from pam.activity import Leg, Plan

# ==============================================================================
# 1. FONCTIONS UTILITAIRES PAM (Tirées de nts.py)
# ==============================================================================

def pam_to_population_no_trips(population: Population) -> pd.DataFrame:
    """
    Extrait les activités de la population PAM en ignorant les trajets.
    C'est la fonction qui correspond à ta logique "continuous activities".
    """
    record = []
    for uid, (_, _, person) in enumerate(population.people()):
        # On parcourt le plan par pas de 2 (Activité, Leg, Activité...)
        # PAM stocke : Act1 -> Leg1 -> Act2 -> Leg2 -> Act3
        # La logique "No Trip" fusionne souvent la fin de l'Act1 avec le début de l'Act2
        
        # NOTE: PAM convertit tout en datetime, on repasse en minutes pour comparer
        def dt_to_min(dt):
            # PAM utilise des objets datetime interne, conversion simple :
            return int(dt.hour * 60 + dt.minute)

        # Itération sur les composants du plan
        for i, component in enumerate(person.plan):
            # On ne garde que les activités
            if not isinstance(component, Leg):
                act_type = component.act
                start = dt_to_min(component.start_time)
                end = dt_to_min(component.end_time)
                
                # Correction PAM : PAM garde parfois les trajets "vides" entre activités
                # Ici on veut juste la liste brute : Act, Start, End
                record.append([person.pid, act_type, start, end])

    df = pd.DataFrame(record, columns=["pid", "act", "start", "end"])
    return df

# ==============================================================================
# 2. ADAPTATEUR EGT -> PAM
# ==============================================================================

def run_pam_pipeline(df_trips_pl, df_persons_pl, df_households_pl):
    print("--- Lancement Pipeline PAM (Reference) ---")
    
    # 1. Conversion Polars -> Pandas (PAM ne gère que Pandas)
    trips = df_trips_pl.to_pandas()
    persons = df_persons_pl.to_pandas()
    households = df_households_pl.to_pandas()

    # 2. Renommage des colonnes pour matcher le standard PAM
    # PAM attend : pid, hid, seq, hzone, ozone, dzone, purp, mode, tst, tet, freq
    rename_map = {
        "person_id": "pid",
        "egt_household_id": "hid", # ou household_id selon ton nettoyage
        "trip_id": "seq",
        "departure_time": "tst",
        "arrival_time": "tet",
        "following_purpose": "purp", # Destination activity
        "preceding_purpose": "oact", # Origin activity (Optionnel mais aide PAM)
        "mode": "mode",
        "origin_departement_id": "ozone",
        "destination_departement_id": "dzone",
        "trip_weight": "freq"
    }
    
    # On applique le mapping uniquement sur les colonnes qui existent
    cols_to_rename = {k: v for k, v in rename_map.items() if k in trips.columns}
    trips = trips.rename(columns=cols_to_rename)
    
    # 3. Conversion des Types et Unités
    # PAM veut des Strings pour les IDs
    trips["pid"] = trips["pid"].astype(str)
    trips["hid"] = trips["hid"].astype(str)
    
    # PAM veut des MINUTES (Int), ton script sort des SECONDES (Float)
    trips["tst"] = (trips["tst"] / 60).fillna(0).astype(int)
    trips["tet"] = (trips["tet"] / 60).fillna(0).astype(int)
    
    # 4. Chargement dans PAM
    # C'est ici que la magie de l'inférence opère
    population = read.load_travel_diary(
        trips=trips,
        persons_attributes=persons.set_index("person_id") if "person_id" in persons.columns else None,
        trip_freq_as_person_freq=True
    )
    
    # 5. Nettoyage standard PAM (Fermeture des boucles, etc.)
    print("PAM : Fix plans...")
    population.fix_plans()
    
    # 6. Export vers format "Schedules" (sans trajets)
    print("PAM : Export vers DataFrame...")
    df_pam_schedules = pam_to_population_no_trips(population)
    
    # Nettoyage final pour comparaison stricte
    df_pam_schedules["act"] = df_pam_schedules["act"].str.lower()
    df_pam_schedules = df_pam_schedules.sort_values(["pid", "start"])
    
    return df_pam_schedules

# ==============================================================================
# 3. VERIFICATION (A exécuter dans ton Main)
# ==============================================================================

if __name__ == "__main__":
    # Supposons que 'loader' et 'cfg' sont déjà initialisés comme dans ton script
    # df_egt_h, df_egt_i, df_egt_t = loader.load_egt()
    
    # 1. Générer la version PAM
    df_pam = run_pam_pipeline(df_egt_t, df_egt_i, df_egt_h)
    
    # 2. Générer TA version Polars
    # (Utilise ta fonction export_trips_to_caveat_csv mais retourne le DF sans sauver)
    # df_custom = export_trips_to_caveat_csv(df_egt_t, "dummy.csv") 
    
    # --- COMPARATEUR ---
    print("\n=== RÉSULTATS DE LA COMPARAISON ===")
    
    # On compare sur un sous-ensemble commun de PIDs (car PAM peut en rejeter certains si plans invalides)
    pids_pam = set(df_pam["pid"].astype(str).unique())
    pids_custom = set(df_custom["pid"].cast(pl.String).unique().to_list())
    
    common_pids = pids_pam.intersection(pids_custom)
    print(f"PIDs communs : {len(common_pids)}")
    print(f"PIDs uniquement dans Custom : {len(pids_custom - pids_pam)} (Probablement rejetés par PAM)")
    
    # On prend un PID au hasard pour voir
    import random
    if common_pids:
        test_pid = random.choice(list(common_pids))
        print(f"\nComparaison pour PID : {test_pid}")
        
        print("\n--- PAM ---")
        print(df_pam[df_pam["pid"] == test_pid][["act", "start", "end"]])
        
        print("\n--- CUSTOM (Polars) ---")
        print(df_custom.filter(pl.col("pid") == int(test_pid)).select(["act", "start", "end"]))

NameError: name 'df_egt_t' is not defined